In [1]:
from pathlib import Path
from datasets import load_dataset

# Chargement explicite des splits
local_data_dir = Path("wmt14_fr_en")
if local_data_dir.exists():
    train_dataset = load_dataset("parquet", data_files=str(local_data_dir / "train*.parquet"), split="train")
    val_dataset = load_dataset("parquet", data_files=str(local_data_dir / "validation*.parquet"), split="train")
    test_dataset = load_dataset("parquet", data_files=str(local_data_dir / "test*.parquet"), split="train")
else:
    train_dataset = load_dataset("wmt14", "fr-en", split="train")
    val_dataset   = load_dataset("wmt14", "fr-en", split="validation")
    test_dataset  = load_dataset("wmt14", "fr-en", split="test")

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/28 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/28 [00:00<?, ?it/s]

In [2]:
%pip install --quiet tensorflow scikit-learn matplotlib optuna pandas datasets sentencepiece

Note: you may need to restart the kernel to use updated packages.


In [3]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import tensorflow as tf
from datasets import load_dataset
from sklearn.model_selection import train_test_split
from tensorflow import keras
from tensorflow.keras import layers

import optuna

SEED = 42
tf.keras.utils.set_random_seed(SEED)
np.random.seed(SEED)

DATASET_NAME = "wmt14"
DATASET_CONFIG = "fr-en"
MAX_VOCAB = 20000
MAX_LEN_FR = 40
MAX_LEN_EN = 40
BATCH_SIZE = 128
OPTUNA_TRIALS = 5
OPTUNA_EPOCHS = 3
FINAL_EPOCHS = 12
SAMPLE_SIZE = 50000
TEST_SIZE = 0.1
VAL_SIZE = 0.1

START_TOKEN = "[start]"
END_TOKEN = "[end]"

In [4]:
train_full = train_dataset.shuffle(seed=SEED)
train_full = train_full.select(range(min(SAMPLE_SIZE, len(train_full))))
train_split = train_full.train_test_split(test_size=VAL_SIZE, seed=SEED)
train_dataset = train_split["train"]
optuna_val_dataset = train_split["test"]
official_val_dataset = val_dataset
official_test_dataset = test_dataset


def extract_pairs(split_dataset):
    source_texts = [row["translation"]["fr"] for row in split_dataset]
    target_texts = [row["translation"]["en"] for row in split_dataset]
    target_texts = [f"{START_TOKEN} {text} {END_TOKEN}" for text in target_texts]
    return source_texts, target_texts


train_sources, train_targets = extract_pairs(train_dataset)
optuna_val_sources, optuna_val_targets = extract_pairs(optuna_val_dataset)
official_val_sources, official_val_targets = extract_pairs(official_val_dataset)
official_test_sources, official_test_targets = extract_pairs(official_test_dataset)

print("Train size:", len(train_sources))
print("Optuna val size:", len(optuna_val_sources))
print("Official val size:", len(official_val_sources))
print("Official test size:", len(official_test_sources))

source_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_LEN_FR,
)

target_vectorizer = keras.layers.TextVectorization(
    max_tokens=MAX_VOCAB,
    standardize="lower_and_strip_punctuation",
    split="whitespace",
    output_mode="int",
    output_sequence_length=MAX_LEN_EN + 2,
)

source_vectorizer.adapt(train_sources)
target_vectorizer.adapt(train_targets)


def vectorize_inputs(source_texts, target_texts):
    source_tokens = source_vectorizer(np.array(source_texts)).numpy().astype("int32")
    target_tokens = target_vectorizer(np.array(target_texts)).numpy().astype("int32")
    decoder_inputs = target_tokens[:, :-1]
    decoder_targets = target_tokens[:, 1:]
    sample_weights = (decoder_targets != 0).astype("float32")
    return source_tokens, decoder_inputs, decoder_targets, sample_weights


X_train, Y_train_in, Y_train_out, W_train = vectorize_inputs(train_sources, train_targets)
X_optuna_val, Y_optuna_val_in, Y_optuna_val_out, W_optuna_val = vectorize_inputs(optuna_val_sources, optuna_val_targets)
X_official_val, Y_official_val_in, Y_official_val_out, W_official_val = vectorize_inputs(official_val_sources, official_val_targets)
X_official_test, Y_official_test_in, Y_official_test_out, W_official_test = vectorize_inputs(official_test_sources, official_test_targets)

print("X_train shape:", X_train.shape)
print("Y_train_in shape:", Y_train_in.shape)
print("Y_train_out shape:", Y_train_out.shape)
print("X_optuna_val shape:", X_optuna_val.shape)
print("X_official_test shape:", X_official_test.shape)

Train size: 45000
Optuna val size: 5000
Official val size: 3000
Official test size: 3003
X_train shape: (45000, 40)
Y_train_in shape: (45000, 41)
Y_train_out shape: (45000, 41)
X_optuna_val shape: (5000, 40)
X_official_test shape: (3003, 40)


In [5]:
def build_seq2seq_gru_model(
    src_vocab_size,
    tgt_vocab_size,
    embedding_dim=128,
    gru_units=256,
    dropout=0.2,
    learning_rate=1e-3,
):
    encoder_inputs = keras.Input(shape=(MAX_LEN_FR,), dtype="int32", name="encoder_inputs")
    encoder_embedding = layers.Embedding(src_vocab_size, embedding_dim, mask_zero=True, name="encoder_embedding")
    encoder_x = encoder_embedding(encoder_inputs)
    _, encoder_state = layers.GRU(
        gru_units,
        return_state=True,
        dropout=dropout,
        name="encoder_gru",
    )(encoder_x)

    decoder_inputs = keras.Input(shape=(MAX_LEN_EN + 1,), dtype="int32", name="decoder_inputs")
    decoder_embedding = layers.Embedding(tgt_vocab_size, embedding_dim, mask_zero=True, name="decoder_embedding")
    decoder_x = decoder_embedding(decoder_inputs)
    decoder_gru = layers.GRU(
        gru_units,
        return_sequences=True,
        return_state=True,
        dropout=dropout,
        name="decoder_gru",
    )
    decoder_outputs, _ = decoder_gru(decoder_x, initial_state=encoder_state)
    decoder_outputs = layers.Dropout(dropout)(decoder_outputs)
    decoder_dense = layers.Dense(tgt_vocab_size, activation="softmax", name="decoder_dense")
    outputs = decoder_dense(decoder_outputs)

    model = keras.Model([encoder_inputs, decoder_inputs], outputs, name="seq2seq_gru_wmt14")
    model.compile(
        optimizer=keras.optimizers.Adam(learning_rate=learning_rate),
        loss="sparse_categorical_crossentropy",
        metrics=["accuracy"],
    )
    return model


def build_inference_models(model):
    encoder_inputs = model.get_layer("encoder_inputs").input
    encoder_embedding = model.get_layer("encoder_embedding")
    encoder_gru = model.get_layer("encoder_gru")
    encoder_x = encoder_embedding(encoder_inputs)
    _, encoder_state = encoder_gru(encoder_x)
    encoder_model = keras.Model(encoder_inputs, encoder_state, name="encoder_model")

    decoder_inputs = keras.Input(shape=(1,), dtype="int32", name="decoder_input_step")
    decoder_state_input = keras.Input(shape=(encoder_gru.units,), name="decoder_state_input")
    decoder_embedding = model.get_layer("decoder_embedding")
    decoder_gru = model.get_layer("decoder_gru")
    decoder_dense = model.get_layer("decoder_dense")

    decoder_x = decoder_embedding(decoder_inputs)
    decoder_outputs, decoder_state = decoder_gru(decoder_x, initial_state=decoder_state_input)
    decoder_outputs = decoder_dense(decoder_outputs)
    decoder_model = keras.Model([decoder_inputs, decoder_state_input], [decoder_outputs, decoder_state], name="decoder_model")
    return encoder_model, decoder_model


def make_greedy_decoder(encoder_model, decoder_model, target_vectorizer):
    vocab = target_vectorizer.get_vocabulary()
    token_to_index = {token: index for index, token in enumerate(vocab)}
    start_index = token_to_index[START_TOKEN]
    end_index = token_to_index[END_TOKEN]

    def decode_sequence(source_text):
        source_tokens = source_vectorizer(np.array([source_text])).numpy().astype("int32")
        state = encoder_model.predict(source_tokens, verbose=0)
        current_token = np.array([[start_index]], dtype="int32")
        decoded_tokens = []

        for _ in range(MAX_LEN_EN + 1):
            token_probs, state = decoder_model.predict([current_token, state], verbose=0)
            next_index = int(np.argmax(token_probs[0, 0]))
            if next_index == 0 or next_index == end_index:
                break
            decoded_tokens.append(vocab[next_index])
            current_token = np.array([[next_index]], dtype="int32")

        return " ".join(decoded_tokens)

    return decode_sequence


source_vocab_size = len(source_vectorizer.get_vocabulary())
target_vocab_size = len(target_vectorizer.get_vocabulary())
print("Source vocab size:", source_vocab_size)
print("Target vocab size:", target_vocab_size)

Source vocab size: 20000
Target vocab size: 20000


In [ ]:
def objective(trial):
    embedding_dim = trial.suggest_categorical("embedding_dim", [64, 128, 256])
    gru_units = trial.suggest_categorical("gru_units", [128, 256, 384])
    dropout = trial.suggest_float("dropout", 0.1, 0.4)
    learning_rate = trial.suggest_float("learning_rate", 1e-4, 3e-3, log=True)

    model = build_seq2seq_gru_model(
        source_vocab_size,
        target_vocab_size,
        embedding_dim=embedding_dim,
        gru_units=gru_units,
        dropout=dropout,
        learning_rate=learning_rate,
    )

    callbacks = [
        keras.callbacks.EarlyStopping(monitor="val_loss", patience=2, restore_best_weights=True),
    ]

    history = model.fit(
        [X_train, Y_train_in],
        Y_train_out,
        sample_weight=W_train,
        validation_data=([X_optuna_val, Y_optuna_val_in], Y_optuna_val_out, W_optuna_val),
        epochs=OPTUNA_EPOCHS,
        batch_size=BATCH_SIZE,
        verbose=0,
        callbacks=callbacks,
    )

    return min(history.history["val_loss"])


use_optuna = True
if use_optuna:
    study = optuna.create_study(direction="minimize")
    study.optimize(objective, n_trials=OPTUNA_TRIALS)
    best_params = study.best_params
else:
    best_params = {
        "embedding_dim": 128,
        "gru_units": 256,
        "dropout": 0.2,
        "learning_rate": 1e-3,
    }

print("Best params:", best_params)

[I 2026-05-20 12:21:48,628] A new study created in memory with name: no-name-1509ef56-0848-41d9-8ce2-fc0093259bc9


[I 2026-05-20 12:21:48,628] A new study created in memory with name: no-name-1509ef56-0848-41d9-8ce2-fc0093259bc9


[I 2026-05-20 12:21:48,628] A new study created in memory with name: no-name-1509ef56-0848-41d9-8ce2-fc0093259bc9


[W 2026-05-20 12:28:50,643] Trial 0 failed with parameters: {'embedding_dim': 128, 'gru_units': 256, 'dropout': 0.14327778399583097, 'learning_rate': 0.0009057826493725895} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\optuna\study\_optimize.py", line 206, in _run_trial
    value_or_values = func(trial)
                      ^^^^^^^^^^^
  File "C:\Users\afagn\AppData\Local\Temp\ipykernel_19156\398417575.py", line 20, in objective
    history = model.fit(
              ^^^^^^^^^^
  File "C:\Users\afagn\AppData\Local\Packages\PythonSoftwareFoundation.Python.3.11_qbz5n2kfra8p0\LocalCache\local-packages\Python311\site-packages\keras\src\utils\traceback_utils.py", line 117, in error_handler
    return fn(*args, **kwargs)
           ^^^^^^^^^^^^^^^^^^^
  File "C:\Users\afagn\AppData\Local\Packages\PythonSo

In [ ]:
final_model = build_seq2seq_gru_model(
    source_vocab_size,
    target_vocab_size,
    **best_params,
)

callbacks = [
    keras.callbacks.EarlyStopping(monitor="val_loss", patience=4, restore_best_weights=True),
    keras.callbacks.ReduceLROnPlateau(monitor="val_loss", factor=0.5, patience=2, min_lr=1e-5),
]

history = final_model.fit(
    [X_train, Y_train_in],
    Y_train_out,
    sample_weight=W_train,
    validation_data=([X_optuna_val, Y_optuna_val_in], Y_optuna_val_out, W_optuna_val),
    epochs=FINAL_EPOCHS,
    batch_size=BATCH_SIZE,
    verbose=1,
    callbacks=callbacks,
)

train_loss, train_accuracy = final_model.evaluate([X_train, Y_train_in], Y_train_out, sample_weight=W_train, verbose=0)
optuna_val_loss, optuna_val_accuracy = final_model.evaluate([X_optuna_val, Y_optuna_val_in], Y_optuna_val_out, sample_weight=W_optuna_val, verbose=0)
official_val_loss, official_val_accuracy = final_model.evaluate([X_official_val, Y_official_val_in], Y_official_val_out, sample_weight=W_official_val, verbose=0)
official_test_loss, official_test_accuracy = final_model.evaluate([X_official_test, Y_official_test_in], Y_official_test_out, sample_weight=W_official_test, verbose=0)

print(f"Train loss:        {train_loss:.4f}, Train accuracy:        {train_accuracy:.4f}")
print(f"Optuna val loss:    {optuna_val_loss:.4f}, Optuna val accuracy:    {optuna_val_accuracy:.4f}")
print(f"Official val loss:  {official_val_loss:.4f}, Official val accuracy:  {official_val_accuracy:.4f}")
print(f"Official test loss: {official_test_loss:.4f}, Official test accuracy: {official_test_accuracy:.4f}")

encoder_model, decoder_model = build_inference_models(final_model)
decode_sequence = make_greedy_decoder(encoder_model, decoder_model, target_vectorizer)

sample_indices = np.linspace(0, len(official_test_sources) - 1, num=min(5, len(official_test_sources)), dtype=int)
for index in sample_indices:
    source_sentence = official_test_sources[index]
    reference_sentence = official_test_targets[index].replace(START_TOKEN, "").replace(END_TOKEN, "").strip()
    predicted_sentence = decode_sequence(source_sentence)
    print("\nFR:", source_sentence)
    print("REF:", reference_sentence)
    print("PRD:", predicted_sentence)